# Complete Pairwise Tanimoto Fold-Distance Plots — Hi Tasks

This notebook generates figures from the fold-distance tables computed in:

`01_tanimoto_distance_tables_hi.ipynb`

No fingerprints, feature importances or distances are recomputed here.

Loaded tables:

- `fold_distance_summary.csv`
- `fold_distance_nn_distributions.csv`
- `fold_distance_activity_detection_vs_random_bits_summary.csv`
- `fold_distance_selected_bits_modelwise.csv`

The main plotted metric is the complete cross-fold pairwise Tanimoto distance. Nearest-neighbour distances are used only as secondary diagnostics.

The figures are model-wise and fold-wise, to avoid averaging out fold-specific effects.

In [ ]:
import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    if start is None:
        start = Path.cwd().resolve()
    current = start
    while current != current.parent:
        if all((current / d).exists() for d in ["data", "results"]):
            return current
        current = current.parent
    raise RuntimeError("Could not find project root containing data/ and results/.")


PROJECT_ROOT = find_project_root()

TASK = "hi"
DATASETS_MAIN = ["drd2", "hiv", "sol"]
DATASETS = DATASETS_MAIN

DATASET_LABELS = {"drd2": "DRD2", "hiv": "HIV", "sol": "Sol"}

SUBSETS = ["F1", "F2", "F3"]
PAIRS = list(combinations(SUBSETS, 2))
PAIR_NAMES = [f"{a}_vs_{b}" for a, b in PAIRS]

PAIR_TO_OUTER_FOLD = {"F1_vs_F2": 1, "F1_vs_F3": 2, "F2_vs_F3": 3}

PAIR_LABELS = {
    "F1_vs_F2": "F1 vs F2",
    "F1_vs_F3": "F1 vs F3",
    "F2_vs_F3": "F2 vs F3",
}

MODELS = ["DT", "LR", "SVM"]

MODEL_LABELS = {
    "DT": "Decision Tree",
    "LR": "Logistic Regression",
    "SVM": "Linear SVM",
}

K_VALUES = [10, 20, 50, 100, 150, 200, 250, 500]

BIT_SOURCE_LABELS = {
    "full_ecfp4": "Full ECFP4",
    "activity_global": "Global activity",
    "activity_ood": "OOD activity",
    "activity_random_shuffle": "Random-shuffle activity",
    "dataset_detection": "Dataset detection",
    "random_bits": "Random bits",
}

PAIR_COLORS = {
    "F1_vs_F2": "#4C78A8",
    "F1_vs_F3": "#F58518",
    "F2_vs_F3": "#54A24B",
}

SOURCE_COLORS = {
    "activity_global": "#2C3E50",
    "activity_ood": "#2563EB",
    "activity_random_shuffle": "#DC2626",
    "dataset_detection": "#7C3AED",
    "random_bits": "#95A5A6",
    "full_ecfp4": "#111827",
}

OUT_ROOT = PROJECT_ROOT / "results" / "results_fold_distance_tanimoto" / TASK
FIG_ROOT = OUT_ROOT / "figures"
FIG_ROOT.mkdir(parents=True, exist_ok=True)

OOD_CROSS_DIR = (
    PROJECT_ROOT / "results" / "results_ood_vs_random_shuffle" / TASK / "cross_dataset"
)

print("Plot notebook setup completed.")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"OUT_ROOT: {OUT_ROOT}")
print(f"FIG_ROOT: {FIG_ROOT}")

In [ ]:
dist_path = OUT_ROOT / "fold_distance_summary.csv"
hist_path = OUT_ROOT / "fold_distance_nn_distributions.csv"
summary_path = OUT_ROOT / "fold_distance_activity_detection_vs_random_bits_summary.csv"
selected_bits_path = OUT_ROOT / "fold_distance_selected_bits_modelwise.csv"


required_paths = [dist_path, summary_path, selected_bits_path]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Missing required table: {path}")

dist_all = pd.read_csv(dist_path)
final_summary = pd.read_csv(summary_path)
selected_bits = pd.read_csv(selected_bits_path)

if hist_path.exists():
    hist_all = pd.read_csv(hist_path)
else:
    hist_all = pd.DataFrame()
    warnings.warn(f"Missing NN histogram table: {hist_path}")

print("Loaded plotting tables.")
print(f"dist_all       : {dist_all.shape}")
print(f"hist_all       : {hist_all.shape}")
print(f"final_summary  : {final_summary.shape}")
print(f"selected_bits  : {selected_bits.shape}")

display(dist_all.head())
display(final_summary.head())
display(selected_bits.head())

In [ ]:
required_dist_cols = [
    "dataset",
    "dataset_label",
    "pair",
    "outer_fold",
    "space",
    "k",
    "model",
    "bit_source",
    "activity_protocol",
    "pairwise_distance",
    "valid_molecule_fraction",
    "valid_pair_fraction",
]

missing = [c for c in required_dist_cols if c not in dist_all.columns]
if missing:
    raise ValueError(f"Missing required columns in dist_all: {missing}")

expected_bit_sources = {
    "full_ecfp4",
    "activity_global",
    "activity_ood",
    "activity_random_shuffle",
    "dataset_detection",
    "random_bits",
}

missing_sources = expected_bit_sources - set(dist_all["bit_source"].unique())
if missing_sources:
    raise ValueError(f"Missing bit sources in dist_all: {missing_sources}")

unexpected_datasets = set(dist_all["dataset"].unique()) - set(DATASETS)
if unexpected_datasets:
    raise ValueError(f"Unexpected datasets in dist_all: {unexpected_datasets}")

assert "kdr" not in set(
    dist_all["dataset"].astype(str).str.lower()
), "KDR leaked into the Tanimoto plot table."

missing_models = set(MODELS) - set(dist_all["model"].unique())
if missing_models:
    raise ValueError(f"Missing models in dist_all: {missing_models}")

observed_k = set(
    dist_all.loc[dist_all["bit_source"] != "full_ecfp4", "k"].astype(int).unique()
)
expected_k = set(K_VALUES)
if observed_k != expected_k:
    raise ValueError(
        f"Unexpected k values. Observed={sorted(observed_k)}, expected={sorted(expected_k)}"
    )

expected_per_dataset = len(MODELS) * len(PAIR_NAMES) * (1 + len(K_VALUES) * (4 + 30))
expected_total = expected_per_dataset * len(DATASETS)

print("Expected row count:")
print(f"  per dataset: {expected_per_dataset}")
print(f"  total:       {expected_total}")
print(f"  observed:    {len(dist_all)}")

if len(dist_all) != expected_total:
    raise ValueError(
        f"Unexpected dist_all row count: got {len(dist_all)}, expected {expected_total}"
    )

if dist_all["pairwise_distance"].notna().sum() == 0:
    raise RuntimeError("All pairwise_distance values are NaN.")

required_selected_cols = [
    "dataset",
    "model",
    "pair",
    "outer_fold",
    "k",
    "bit_source",
    "bits_json",
]

missing_selected = [c for c in required_selected_cols if c not in selected_bits.columns]
if missing_selected:
    raise ValueError(f"Missing columns in selected_bits: {missing_selected}")

print("Sanity checks passed.")
print(
    f"Finite pairwise_distance: "
    f"{dist_all['pairwise_distance'].notna().sum()}/{len(dist_all)}"
)

In [ ]:
def set_plot_style():
    mpl.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "axes.titlesize": 10.5,
            "axes.labelsize": 9.5,
            "xtick.labelsize": 8.2,
            "ytick.labelsize": 8.2,
            "legend.fontsize": 8,
            "figure.dpi": 150,
            "savefig.dpi": 600,
            "axes.linewidth": 0.8,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )


def save_fig(fig, name: str):
    path = FIG_ROOT / name
    fig.savefig(f"{path}.png", dpi=600, bbox_inches="tight")
    fig.savefig(f"{path}.pdf", bbox_inches="tight")
    print(f"Saved: {path}.png")
    print(f"Saved: {path}.pdf")


def source_label(src: str) -> str:
    return BIT_SOURCE_LABELS.get(src, src)


def model_label(model: str) -> str:
    return MODEL_LABELS.get(model, model)


def pair_label(pair: str) -> str:
    return PAIR_LABELS.get(pair, pair)


def finite_mean(x):
    x = pd.to_numeric(x, errors="coerce")
    return float(np.nanmean(x)) if x.notna().any() else np.nan


set_plot_style()
print("Plot style configured.")

In [ ]:
def aggregate_source(
    df: pd.DataFrame,
    dataset: str,
    model: str,
    pair: str,
    bit_source: str,
    metric: str,
) -> pd.DataFrame:
    sub = df[
        (df["dataset"] == dataset)
        & (df["model"] == model)
        & (df["pair"] == pair)
        & (df["bit_source"] == bit_source)
        & (df["k"].isin(K_VALUES))
    ].copy()

    if sub.empty:
        return pd.DataFrame(columns=["k", metric])

    out = sub.groupby("k", as_index=False)[metric].mean().sort_values("k")

    return out


def full_reference(
    df: pd.DataFrame,
    dataset: str,
    model: str,
    pair: str,
    metric: str = "pairwise_distance",
) -> float:
    sub = df[
        (df["dataset"] == dataset)
        & (df["model"] == model)
        & (df["pair"] == pair)
        & (df["bit_source"] == "full_ecfp4")
    ]
    if sub.empty:
        return np.nan
    return finite_mean(sub[metric])

# Activity OOD vs random-shuffle distance

Fold-wise complete pairwise Tanimoto distance using activity-relevant top-k features.

Rows are datasets, columns are the two activity protocols, and each line is a fold pair.

In [ ]:
def fig_activity_ood_vs_random_distance(dist_all: pd.DataFrame, model: str):
    set_plot_style()

    sources = ["activity_ood", "activity_random_shuffle"]
    n_rows = len(DATASETS)
    n_cols = len(sources)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.6 * n_cols, 3.0 * n_rows),
        sharex=True,
        sharey=False,
        squeeze=False,
    )

    for i, ds in enumerate(DATASETS):
        for j, src in enumerate(sources):
            ax = axes[i, j]

            for pair in PAIR_NAMES:
                sub = aggregate_source(
                    dist_all,
                    dataset=ds,
                    model=model,
                    pair=pair,
                    bit_source=src,
                    metric="pairwise_distance",
                )

                if sub.empty:
                    continue

                ax.plot(
                    sub["k"],
                    sub["pairwise_distance"],
                    marker="o",
                    lw=1.7,
                    color=PAIR_COLORS.get(pair, "C0"),
                    label=pair_label(pair),
                )

                full_val = full_reference(dist_all, ds, model, pair)
                if np.isfinite(full_val):
                    ax.axhline(
                        full_val,
                        ls=":",
                        lw=0.9,
                        color=PAIR_COLORS.get(pair, "C0"),
                        alpha=0.55,
                    )

            ax.set_xscale("log")
            ax.set_xticks(K_VALUES)
            ax.set_xticklabels([str(k) for k in K_VALUES])
            ax.grid(ls="--", alpha=0.35)

            if i == 0:
                ax.set_title(source_label(src), fontweight="bold")
            if j == 0:
                ax.set_ylabel(
                    f"{DATASET_LABELS.get(ds, ds)}\nPairwise distance",
                    fontweight="bold",
                )
            if i == n_rows - 1:
                ax.set_xlabel("Top-k ECFP4 bits")

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 1.01),
    )

    fig.suptitle(
        f"Activity-restricted fold distance vs k — {model_label(model)}",
        y=1.055,
        fontweight="bold",
        fontsize=13,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.98])

    save_fig(fig, f"figA_activity_ood_vs_random_distance_{model.lower()}")
    plt.show()
    plt.close(fig)


for model in MODELS:
    fig_activity_ood_vs_random_distance(dist_all, model)

## Dataset-detection top-k distance

Distance induced by the top-k features from the fold-discriminator task.

In [ ]:
def fig_dataset_detection_distance(dist_all: pd.DataFrame):
    set_plot_style()

    n_rows = len(DATASETS)
    n_cols = len(MODELS)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.2 * n_cols, 3.0 * n_rows),
        sharex=True,
        sharey=False,
        squeeze=False,
    )

    for i, ds in enumerate(DATASETS):
        for j, model in enumerate(MODELS):
            ax = axes[i, j]

            for pair in PAIR_NAMES:
                sub = aggregate_source(
                    dist_all,
                    dataset=ds,
                    model=model,
                    pair=pair,
                    bit_source="dataset_detection",
                    metric="pairwise_distance",
                )

                if sub.empty:
                    continue

                ax.plot(
                    sub["k"],
                    sub["pairwise_distance"],
                    marker="o",
                    lw=1.7,
                    color=PAIR_COLORS.get(pair, "C0"),
                    label=pair_label(pair),
                )

                full_val = full_reference(dist_all, ds, model, pair)
                if np.isfinite(full_val):
                    ax.axhline(
                        full_val,
                        ls=":",
                        lw=0.9,
                        color=PAIR_COLORS.get(pair, "C0"),
                        alpha=0.55,
                    )

            ax.set_xscale("log")
            ax.set_xticks(K_VALUES)
            ax.set_xticklabels([str(k) for k in K_VALUES])
            ax.grid(ls="--", alpha=0.35)

            if i == 0:
                ax.set_title(model_label(model), fontweight="bold")
            if j == 0:
                ax.set_ylabel(
                    f"{DATASET_LABELS.get(ds, ds)}\nPairwise distance",
                    fontweight="bold",
                )
            if i == n_rows - 1:
                ax.set_xlabel("Top-k ECFP4 bits")

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 1.01),
    )

    fig.suptitle(
        "Dataset-detection restricted fold distance vs k",
        y=1.055,
        fontweight="bold",
        fontsize=13,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.98])

    save_fig(fig, "figB_dataset_detection_distance_vs_k")
    plt.show()
    plt.close(fig)


fig_dataset_detection_distance(dist_all)

## Distance relative to random-bit baseline

Restricted-space distance minus random-bit distance, computed fold-wise and model-wise.

In [ ]:
def build_delta_vs_random(dist_all: pd.DataFrame) -> pd.DataFrame:
    sources = ["activity_ood", "activity_random_shuffle", "dataset_detection"]

    rb = (
        dist_all[dist_all["bit_source"] == "random_bits"]
        .groupby(
            ["dataset", "model", "pair", "outer_fold", "k"],
            as_index=False,
        )["pairwise_distance"]
        .mean()
        .rename(columns={"pairwise_distance": "random_bits_distance"})
    )

    rows = []

    for src in sources:
        sub = dist_all[dist_all["bit_source"] == src].copy()

        sub = (
            sub.groupby(
                ["dataset", "dataset_label", "model", "pair", "outer_fold", "k"],
                as_index=False,
            )["pairwise_distance"]
            .mean()
            .rename(columns={"pairwise_distance": "restricted_distance"})
        )

        merged = sub.merge(
            rb,
            on=["dataset", "model", "pair", "outer_fold", "k"],
            how="left",
        )

        merged["bit_source"] = src
        merged["delta_vs_random_bits"] = (
            merged["restricted_distance"] - merged["random_bits_distance"]
        )

        rows.append(merged)

    out = pd.concat(rows, ignore_index=True)
    out_path = OUT_ROOT / "fold_distance_delta_vs_random_bits_for_plots.csv"
    out.to_csv(out_path, index=False)

    print(f"Saved delta table: {out_path} ({len(out)} rows)")
    return out


delta_random = build_delta_vs_random(dist_all)
display(delta_random.head())

In [ ]:
def fig_delta_vs_random_by_model(delta_random: pd.DataFrame, model: str):
    set_plot_style()

    sources = ["activity_ood", "activity_random_shuffle", "dataset_detection"]
    n_rows = len(DATASETS)
    n_cols = len(sources)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.4 * n_cols, 3.0 * n_rows),
        sharex=True,
        sharey=True,
        squeeze=False,
    )

    for i, ds in enumerate(DATASETS):
        for j, src in enumerate(sources):
            ax = axes[i, j]

            for pair in PAIR_NAMES:
                sub = delta_random[
                    (delta_random["dataset"] == ds)
                    & (delta_random["model"] == model)
                    & (delta_random["pair"] == pair)
                    & (delta_random["bit_source"] == src)
                ].sort_values("k")

                if sub.empty:
                    continue

                ax.plot(
                    sub["k"],
                    sub["delta_vs_random_bits"],
                    marker="o",
                    lw=1.7,
                    color=PAIR_COLORS.get(pair, "C0"),
                    label=pair_label(pair),
                )

            ax.axhline(0, color="black", lw=0.9, ls="--", alpha=0.8)
            ax.set_xscale("log")
            ax.set_xticks(K_VALUES)
            ax.set_xticklabels([str(k) for k in K_VALUES])
            ax.grid(ls="--", alpha=0.35)

            if i == 0:
                ax.set_title(source_label(src), fontweight="bold")
            if j == 0:
                ax.set_ylabel(
                    f"{DATASET_LABELS.get(ds, ds)}\nΔ distance",
                    fontweight="bold",
                )
            if i == n_rows - 1:
                ax.set_xlabel("Top-k ECFP4 bits")

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 1.01),
    )

    fig.suptitle(
        f"Restricted distance minus random-bit baseline — {model_label(model)}",
        y=1.055,
        fontweight="bold",
        fontsize=13,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.98])

    save_fig(fig, f"figC_delta_vs_random_bits_{model.lower()}")
    plt.show()
    plt.close(fig)


for model in MODELS:
    fig_delta_vs_random_by_model(delta_random, model)

## Restricted-space coverage

Coverage is measured by valid molecule fraction and valid pair fraction.

In [ ]:
def fig_coverage_by_model(
    dist_all: pd.DataFrame,
    model: str,
    metric: str = "valid_molecule_fraction",
):
    set_plot_style()

    sources = [
        "activity_ood",
        "activity_random_shuffle",
        "dataset_detection",
        "random_bits",
    ]

    n_rows = len(DATASETS)
    n_cols = len(sources)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.1 * n_cols, 2.8 * n_rows),
        sharex=True,
        sharey=True,
        squeeze=False,
    )

    for i, ds in enumerate(DATASETS):
        for j, src in enumerate(sources):
            ax = axes[i, j]

            for pair in PAIR_NAMES:
                sub = aggregate_source(
                    dist_all,
                    dataset=ds,
                    model=model,
                    pair=pair,
                    bit_source=src,
                    metric=metric,
                )

                if sub.empty:
                    continue

                ax.plot(
                    sub["k"],
                    sub[metric],
                    marker="o",
                    lw=1.5,
                    color=PAIR_COLORS.get(pair, "C0"),
                    label=pair_label(pair),
                )

            ax.set_xscale("log")
            ax.set_xticks(K_VALUES)
            ax.set_xticklabels([str(k) for k in K_VALUES])
            ax.set_ylim(-0.02, 1.02)
            ax.grid(ls="--", alpha=0.35)

            if i == 0:
                ax.set_title(source_label(src), fontweight="bold")
            if j == 0:
                ax.set_ylabel(
                    f"{DATASET_LABELS.get(ds, ds)}\n{metric}",
                    fontweight="bold",
                )
            if i == n_rows - 1:
                ax.set_xlabel("Top-k ECFP4 bits")

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 1.01),
    )

    title_map = {
        "valid_molecule_fraction": "Valid molecule fraction",
        "valid_pair_fraction": "Valid pair fraction",
    }

    fig.suptitle(
        f"{title_map.get(metric, metric)} vs k — {model_label(model)}",
        y=1.055,
        fontweight="bold",
        fontsize=13,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.98])

    save_fig(fig, f"figD_{metric}_{model.lower()}")
    plt.show()
    plt.close(fig)


for model in MODELS:
    fig_coverage_by_model(dist_all, model, metric="valid_molecule_fraction")
    fig_coverage_by_model(dist_all, model, metric="valid_pair_fraction")

## Feature-overlap tables

Overlap is computed from the selected top-k bit sets saved by the table notebook.

In [ ]:
def parse_bits(x) -> set[int]:
    if pd.isna(x):
        return set()
    vals = json.loads(x)
    return set(int(v) for v in vals)


def build_overlap_table(selected_bits: pd.DataFrame) -> pd.DataFrame:
    comparisons = [
        (
            "activity_ood",
            "activity_random_shuffle",
            "OOD activity vs random-shuffle activity",
        ),
        (
            "activity_ood",
            "dataset_detection",
            "OOD activity vs dataset detection",
        ),
        (
            "activity_random_shuffle",
            "dataset_detection",
            "Random-shuffle activity vs dataset detection",
        ),
        (
            "activity_global",
            "dataset_detection",
            "Global activity vs dataset detection",
        ),
    ]

    rows = []

    keys = ["dataset", "dataset_label", "model", "pair", "outer_fold", "k"]

    for key_vals, group in selected_bits.groupby(keys, dropna=False):
        key_dict = dict(zip(keys, key_vals))
        bit_map = {
            row["bit_source"]: parse_bits(row["bits_json"])
            for _, row in group.iterrows()
        }

        for src_a, src_b, label in comparisons:
            a = bit_map.get(src_a, set())
            b = bit_map.get(src_b, set())

            inter = a & b
            union = a | b

            k = int(key_dict["k"])
            expected_random_overlap = k / 2048.0 if k > 0 else np.nan

            overlap_fraction = len(inter) / max(k, 1)
            jaccard = len(inter) / len(union) if len(union) > 0 else np.nan
            enrichment = (
                overlap_fraction / expected_random_overlap
                if expected_random_overlap and expected_random_overlap > 0
                else np.nan
            )

            rows.append(
                {
                    **key_dict,
                    "comparison": f"{src_a}__vs__{src_b}",
                    "comparison_label": label,
                    "source_a": src_a,
                    "source_b": src_b,
                    "n_a": len(a),
                    "n_b": len(b),
                    "overlap_count": len(inter),
                    "overlap_fraction": overlap_fraction,
                    "jaccard": jaccard,
                    "overlap_enrichment": enrichment,
                }
            )

    out = pd.DataFrame(rows)

    out_path = OUT_ROOT / "fold_distance_selected_bits_overlap_modelwise.csv"
    out.to_csv(out_path, index=False)

    print(f"Saved overlap table: {out_path} ({len(out)} rows)")
    return out


overlap_table = build_overlap_table(selected_bits)
display(overlap_table.head())

## OOD activity vs random-shuffle activity overlap

This plot checks whether the two activity-selection protocols rely on similar top-k ECFP4 bits.

In [ ]:
def fig_overlap_grid(
    overlap_table: pd.DataFrame,
    comparison: str,
    metric: str = "overlap_fraction",
    fname_suffix: str | None = None,
):
    set_plot_style()

    sub_all = overlap_table[overlap_table["comparison"] == comparison].copy()

    if sub_all.empty:
        print(f"No overlap rows for comparison={comparison}")
        return

    n_rows = len(DATASETS)
    n_cols = len(MODELS)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.2 * n_cols, 3.0 * n_rows),
        sharex=True,
        sharey=False,
        squeeze=False,
    )

    for i, ds in enumerate(DATASETS):
        for j, model in enumerate(MODELS):
            ax = axes[i, j]

            for pair in PAIR_NAMES:
                sub = sub_all[
                    (sub_all["dataset"] == ds)
                    & (sub_all["model"] == model)
                    & (sub_all["pair"] == pair)
                ].sort_values("k")

                if sub.empty:
                    continue

                ax.plot(
                    sub["k"],
                    sub[metric],
                    marker="o",
                    lw=1.7,
                    color=PAIR_COLORS.get(pair, "C0"),
                    label=pair_label(pair),
                )

            ax.set_xscale("log")
            ax.set_xticks(K_VALUES)
            ax.set_xticklabels([str(k) for k in K_VALUES])
            ax.grid(ls="--", alpha=0.35)

            if metric in {"overlap_fraction", "jaccard"}:
                ax.set_ylim(-0.02, 1.02)

            if i == 0:
                ax.set_title(model_label(model), fontweight="bold")
            if j == 0:
                ax.set_ylabel(
                    f"{DATASET_LABELS.get(ds, ds)}\n{metric}",
                    fontweight="bold",
                )
            if i == n_rows - 1:
                ax.set_xlabel("Top-k ECFP4 bits")

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 1.01),
    )

    title = sub_all["comparison_label"].iloc[0]
    fig.suptitle(
        f"{title} — {metric}",
        y=1.055,
        fontweight="bold",
        fontsize=13,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.98])

    if fname_suffix is None:
        fname_suffix = comparison.replace("__vs__", "_vs_")

    save_fig(fig, f"figE_overlap_{fname_suffix}_{metric}")
    plt.show()
    plt.close(fig)


fig_overlap_grid(
    overlap_table,
    comparison="activity_ood__vs__activity_random_shuffle",
    metric="overlap_fraction",
    fname_suffix="activity_ood_vs_random_shuffle",
)

fig_overlap_grid(
    overlap_table,
    comparison="activity_ood__vs__activity_random_shuffle",
    metric="jaccard",
    fname_suffix="activity_ood_vs_random_shuffle",
)

## Activity features vs dataset-detection features

These plots measure whether activity-relevant bits overlap with the fold-discriminator bits.

In [ ]:
fig_overlap_grid(
    overlap_table,
    comparison="activity_ood__vs__dataset_detection",
    metric="overlap_fraction",
    fname_suffix="activity_ood_vs_dataset_detection",
)

fig_overlap_grid(
    overlap_table,
    comparison="activity_ood__vs__dataset_detection",
    metric="overlap_enrichment",
    fname_suffix="activity_ood_vs_dataset_detection",
)

fig_overlap_grid(
    overlap_table,
    comparison="activity_random_shuffle__vs__dataset_detection",
    metric="overlap_fraction",
    fname_suffix="activity_random_shuffle_vs_dataset_detection",
)

fig_overlap_grid(
    overlap_table,
    comparison="activity_random_shuffle__vs__dataset_detection",
    metric="overlap_enrichment",
    fname_suffix="activity_random_shuffle_vs_dataset_detection",
)

## Big fold-wise summary table

This table joins distances, random-bit deltas, coverage and selected-feature overlaps.

In [ ]:
def build_big_summary_table(
    final_summary: pd.DataFrame,
    overlap_table: pd.DataFrame,
) -> pd.DataFrame:
    base_cols = ["dataset", "dataset_label", "model", "pair", "outer_fold", "k"]

    dist_wide = final_summary.pivot_table(
        index=base_cols,
        columns="bit_source",
        values="pairwise_distance_mean",
        aggfunc="mean",
    ).reset_index()

    dist_wide = dist_wide.rename(
        columns={
            "full_ecfp4": "distance_full_ecfp4",
            "activity_global": "distance_activity_global",
            "activity_ood": "distance_activity_ood",
            "activity_random_shuffle": "distance_activity_random_shuffle",
            "dataset_detection": "distance_dataset_detection",
            "random_bits": "distance_random_bits",
        }
    )

    coverage_wide = final_summary.pivot_table(
        index=base_cols,
        columns="bit_source",
        values="valid_molecule_fraction_mean",
        aggfunc="mean",
    ).reset_index()

    coverage_wide = coverage_wide.rename(
        columns={
            "activity_ood": "coverage_activity_ood",
            "activity_random_shuffle": "coverage_activity_random_shuffle",
            "dataset_detection": "coverage_dataset_detection",
            "random_bits": "coverage_random_bits",
        }
    )

    keep_cov = base_cols + [
        c for c in coverage_wide.columns if c.startswith("coverage_")
    ]

    out = dist_wide.merge(
        coverage_wide[keep_cov],
        on=base_cols,
        how="left",
    )

    for src in [
        "activity_global",
        "activity_ood",
        "activity_random_shuffle",
        "dataset_detection",
    ]:
        dist_col = f"distance_{src}"
        if dist_col in out.columns and "distance_random_bits" in out.columns:
            out[f"delta_{src}_minus_random_bits"] = (
                out[dist_col] - out["distance_random_bits"]
            )

    overlap_keep = overlap_table[
        overlap_table["comparison"].isin(
            [
                "activity_ood__vs__activity_random_shuffle",
                "activity_ood__vs__dataset_detection",
                "activity_random_shuffle__vs__dataset_detection",
            ]
        )
    ].copy()

    overlap_frac = overlap_keep.pivot_table(
        index=base_cols,
        columns="comparison",
        values="overlap_fraction",
        aggfunc="mean",
    ).reset_index()

    overlap_frac = overlap_frac.rename(
        columns={
            "activity_ood__vs__activity_random_shuffle": "overlap_activity_ood_vs_random_shuffle",
            "activity_ood__vs__dataset_detection": "overlap_activity_ood_vs_dataset_detection",
            "activity_random_shuffle__vs__dataset_detection": "overlap_activity_random_shuffle_vs_dataset_detection",
        }
    )

    out = out.merge(overlap_frac, on=base_cols, how="left")

    out = out.sort_values(
        ["dataset", "model", "outer_fold", "k"],
        na_position="last",
    ).reset_index(drop=True)

    out_path = OUT_ROOT / "fold_distance_big_summary_for_plots.csv"
    out.to_csv(out_path, index=False)

    print(f"Saved big summary table: {out_path} ({len(out)} rows)")
    return out


big_summary = build_big_summary_table(final_summary, overlap_table)
display(big_summary.head(20))

In [ ]:
print("Generated tables:")
for p in [
    OUT_ROOT / "fold_distance_delta_vs_random_bits_for_plots.csv",
    OUT_ROOT / "fold_distance_selected_bits_overlap_modelwise.csv",
    OUT_ROOT / "fold_distance_big_summary_for_plots.csv",
]:
    print(f"  {p} | exists={p.exists()}")

print("\nFigures saved in:")
print(FIG_ROOT)

figs = sorted(FIG_ROOT.glob("*.png"))
print(f"\nPNG figures: {len(figs)}")
for p in figs:
    print(f"  {p.name}")